# 11 — Text Processing & String Vectorization (`.str` Accessor)
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, NLP Preprocessing, and Data Cleaning Interviews.*

---

## 📌 Executive Summary & Interview Expectations
Real-world data is predominantly unstructured or semi-structured text. In technical interviews, interviewers test whether you can manipulate text **in vectorized C/Arrow speed** using the `.str` accessor, rather than falling back to slow Python `for` loops or unvectorized list comprehensions.

### Core Competencies Tested in this Module:
1. **The `.str` Accessor Architecture**: Why `.str` methods safely propagate `NaN` without raising exceptions.
2. **Whitespace Trimming & Case Normalization**: `.strip()`, `.lower()`, `.title()`, and clean pipeline chaining.
3. **Vectorized Slicing & Indexing**: Slicing strings with `.str[start:stop]` and extracting elements with `.str.get()`.
4. **Splitting & Expansion**: `.str.split(pat, expand=True)` to convert delimited strings directly into DataFrame columns.
5. **Regular Expressions in Pandas**: Regex replacement, regex extraction (`.str.extract()`), and avoiding the `regex=True` warning.
6. **Modern String Dtypes**: PyArrow strings (`string[pyarrow]`) vs legacy NumPy `object` strings.
7. **Interview Corner**: The `expand=True` mechanics, regex extraction drills, and parsing complex unstructured addresses.

## 1. Environment Setup & Data Ingestion
We load both `chicago_food_inspections.csv` and `customers.csv` with automated remote fallbacks.

In [1]:
import os
import re
import numpy as np
import pandas as pd

# Load Chicago Food Inspections dataset
insp_path = "chicago_food_inspections.csv"
if not os.path.exists(insp_path):
    insp_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_06_working_with_text_data/chicago_food_inspections.csv"

inspections = pd.read_csv(insp_path)
print("Inspections dataset loaded. Shape:", inspections.shape)
inspections.head(4)

Inspections dataset loaded. Shape: (153810, 2)


,Name,Risk
0,MARRIOT MARQUIS CHICAGO,Risk 1 (High)
1,JETS PIZZA,Risk 2 (Medium)
2,ROOM 1520,Risk 3 (Low)
3,MARRIOT MARQUIS CHICAGO,Risk 1 (High)


## 2. Whitespace Trimming & Case Normalization

### ⚠️ Top Interview Note: Why `.str` Accessor is Safe on Missing Data
- In standard Python: `'  test  '.strip()` works, but `None.strip()` raises an `AttributeError`.
- In Pandas: `s.str.strip()` automatically ignores `NaN` values and propagates them safely without crashing!

In [2]:
# Notice leading and trailing whitespace in restaurant names
print("Raw sample name with whitespace:", repr(inspections["Name"].iloc[0]))

# Vectorized strip across the entire column
inspections["Name"] = inspections["Name"].str.strip()
print("Cleaned sample name:           ", repr(inspections["Name"].iloc[0]))

Raw sample name with whitespace: ' MARRIOT MARQUIS CHICAGO   '
Cleaned sample name:            'MARRIOT MARQUIS CHICAGO'


In [3]:
# Case transformations
print("Lowercase:  ", inspections["Name"].str.lower().iloc[0])
print("Title Case: ", inspections["Name"].str.title().iloc[0])
print("Uppercase:  ", inspections["Name"].str.upper().iloc[0])

Lowercase:   marriot marquis chicago
Title Case:  Marriot Marquis Chicago
Uppercase:   MARRIOT MARQUIS CHICAGO


## 3. String Slicing & Character Replacement

### ⚠️ Top Interview Question: Python Slice Syntax in Pandas
Pandas supports bracket slicing directly on the `.str` accessor:
- `s.str[start:stop:step]`
- Slicing negative offsets: `s.str[:-1]` (all characters except the last).

In [4]:
# Clean Risk column by dropping nulls
inspections = inspections.dropna(subset=["Risk"]).copy()
print("Unique Risk labels:\n", inspections["Risk"].unique())

Unique Risk labels:
 <ArrowStringArray>
['Risk 1 (High)', 'Risk 2 (Medium)', 'Risk 3 (Low)', 'All']
Length: 4, dtype: str


In [5]:
# Slicing the Risk number (e.g. 'Risk 1 (High)' -> '1')
risk_num = inspections["Risk"].str.slice(5, 6)
# Equivalent bracket notation:
risk_num_bracket = inspections["Risk"].str[5:6]

# Slicing the category description (e.g. 'High', 'Medium', 'Low')
risk_category = inspections["Risk"].str[8:-1]

inspections["Risk_Level"] = risk_num
inspections["Risk_Category"] = risk_category
inspections[["Risk", "Risk_Level", "Risk_Category"]].head()

,Risk,Risk_Level,Risk_Category
0,Risk 1 (High),1,High
1,Risk 2 (Medium),2,Medium
2,Risk 3 (Low),3,Low
3,Risk 1 (High),1,High
4,Risk 1 (High),1,High


## 4. Substring Searching: `.contains()`, `.startswith()`, `.endswith()`

> 💡 **Interview Pro-Tip**:
> In text searches, always chain `.str.lower()` before `.contains()` for case-insensitive matching, or pass `case=False` directly: `s.str.contains('pizza', case=False, na=False)`.

In [6]:
# Case-insensitive substring search for 'pizza'
has_pizza = inspections["Name"].str.contains("pizza", case=False, na=False)
print(f"Total restaurants with 'pizza' in their name: {has_pizza.sum()}")
inspections[has_pizza].head(3)

Total restaurants with 'pizza' in their name: 3992


,Name,Risk,Risk_Level,Risk_Category
1,JETS PIZZA,Risk 2 (Medium),2,Medium
19,NANCY'S HOME OF STUFFED PIZZA,Risk 1 (High),1,High
27,"NARY'S GRILL & PIZZA ,INC.",Risk 1 (High),1,High


In [7]:
# Prefix and suffix filtering
starts_taco = inspections["Name"].str.startswith("TACO", na=False)
ends_grill = inspections["Name"].str.endswith("GRILL", na=False)

print(f"Starts with 'TACO': {starts_taco.sum()}")
print(f"Ends with 'GRILL':   {ends_grill.sum()}")

Starts with 'TACO': 697
Ends with 'GRILL':   2955


## 5. Splitting Strings: `expand=True` Mechanics

### ⚠️ Top Interview Question: `expand=False` vs `expand=True`
- **`expand=False` (default)**: Returns a **Series of Python lists**.
- **`expand=True`**: Unpacks the split elements into a **multi-column `pd.DataFrame`**!
  This allows immediate assignment back to multiple DataFrame columns:
  ```python
  df[['First', 'Last']] = df['Full_Name'].str.split(' ', n=1, expand=True)
  ```

In [8]:
# Load Customers dataset
cust_path = "customers.csv"
if not os.path.exists(cust_path):
    cust_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_06_working_with_text_data/customers.csv"

customers = pd.read_csv(cust_path)
print("Customers dataset loaded. Shape:", customers.shape)
customers.head(3)

Customers dataset loaded. Shape: (9961, 2)


,Name,Address
0,Frank Manning,"6461 Quinn Groves, East Matthew, New Hampshire..."
1,Elizabeth Johnson,"1360 Tracey Ports Apt. 419, Kyleport, Vermont,..."
2,Donald Stephens,"19120 Fleming Manors, Prestonstad, Montana, 23495"


In [9]:
# Demonstrating expand=False (Series of lists) vs expand=True (DataFrame)
split_series = customers["Name"].str.split(" ", n=1, expand=False)
split_df = customers["Name"].str.split(" ", n=1, expand=True)

print("expand=False type:", type(split_series))
display(split_series.head(2))

print("\nexpand=True type: ", type(split_df))
display(split_df.head(2))

expand=False type: <class 'pandas.Series'>


0        [Frank, Manning]
1    [Elizabeth, Johnson]
Name: Name, dtype: object


expand=True type:  <class 'pandas.DataFrame'>


,0,1
0,Frank,Manning
1,Elizabeth,Johnson


In [10]:
# Idiomatic multi-column unpacking with expand=True
customers[["First Name", "Last Name"]] = customers["Name"].str.split(" ", n=1, expand=True)

# Split address into Street, City, State, and Zip
customers[["Street", "City", "State", "Zip"]] = customers["Address"].str.split(", ", expand=True)

# Drop original combined columns
customers = customers.drop(columns=["Name", "Address"])
customers.head()

,First Name,Last Name,Street,City,State,Zip
0,Frank,Manning,6461 Quinn Groves,East Matthew,New Hampshire,16656
1,Elizabeth,Johnson,1360 Tracey Ports Apt. 419,Kyleport,Vermont,31924
2,Donald,Stephens,19120 Fleming Manors,Prestonstad,Montana,23495
3,Michael,Vincent III,441 Olivia Creek,Jimmymouth,Georgia,82991
4,Jasmine,Zamora,4246 Chelsey Ford Apt. 310,Karamouth,Utah,76252


## 6. Regular Expressions in Pandas: `.replace()` and `.extract()`

### 💡 Interview Pro-Tip — The `regex=False` Deprecation in Pandas 2.x
In older Pandas versions, `str.replace("a", "b")` defaulted to regex matching. In modern Pandas:
- If passing a regex, explicitly pass **`regex=True`**.
- If replacing literal strings, pass **`regex=False`** for a 5x speedup!

In [11]:
# Mask street numbers with asterisks using regex
masked_streets = customers["Street"].str.replace(r"^\d+", "****", regex=True)
print("Original Streets:\n", customers["Street"].head(3))
print("\nMasked Streets:\n", masked_streets.head(3))

Original Streets:
 0             6461 Quinn Groves
1    1360 Tracey Ports Apt. 419
2          19120 Fleming Manors
Name: Street, dtype: str

Masked Streets:
 0             **** Quinn Groves
1    **** Tracey Ports Apt. 419
2           **** Fleming Manors
Name: Street, dtype: str


## 7. String Vectorization Cheat Sheet

| Task | Idiomatic Syntax | Key Parameter |
| :--- | :--- | :--- |
| **Strip Whitespace** | `s.str.strip()` | Null-safe |
| **Case-Insensitive Search** | `s.str.contains('pat', case=False, na=False)` | `na=False` prevents boolean errors |
| **Unpack to Columns** | `s.str.split('delim', expand=True)` | Generates 2D DataFrame directly |
| **Limit Splits** | `s.str.split(' ', n=1, expand=True)` | Splits only on first occurrence |
| **Slice Bracket** | `s.str[start:stop]` | Pythonic slice notation |
| **Regex Extraction** | `s.str.extract(r'(pattern)')` | Requires capture group `()` |

---
## 🎯 8. Technical Interview Corner: Tricky Questions & Drills

### Q1: PyArrow Strings (`string[pyarrow]`) vs Python Object Strings
**Question**: What is the difference between storing strings as `object` vs `string[pyarrow]` in modern Pandas?

**Answer**:
- **`object` dtype**: Stores an array of 64-bit pointers pointing to individual Python `str` heap objects scattered across memory. Causes memory fragmentation, GIL contention, and cache misses.
- **`string[pyarrow]`**: Stores strings in contiguous Apache Arrow memory buffers (lengths buffer + contiguous bytes buffer).
  - Uses **~70% less memory**.
  - Operates across CPU SIMD lanes and multi-threaded cores without Python GIL locking, running **5x to 10x faster**.

In [12]:
# Comparing object vs pyarrow string memory
obj_str = customers["Street"].copy()
arrow_str = obj_str.astype("string[pyarrow]")

print(f"NumPy Object memory:  {obj_str.memory_usage(deep=True):,} bytes")
print(f"PyArrow String memory: {arrow_str.memory_usage(deep=True):,} bytes")
print(f"Memory reduction:      {((obj_str.memory_usage(deep=True) - arrow_str.memory_usage(deep=True))/obj_str.memory_usage(deep=True))*100:.1f}%")

NumPy Object memory:  304,240 bytes
PyArrow String memory: 304,240 bytes
Memory reduction:      0.0%


### Q2: Structured Regex Extraction with `.str.extract()`
**Question**: An interviewer asks: *"You have a column of messy strings containing order codes like `ORD-9821-NY`. How do you extract the order number and state into separate columns in a single vectorized step?"*

**Answer**:
Use **`.str.extract(r'regex_with_named_groups')`**!
Named capture groups `(?P<col_name>pattern)` automatically become column headers in the resulting DataFrame.

In [13]:
# Demonstration of named capture groups with .str.extract()
codes = pd.Series(["ORD-9821-NY", "ORD-1044-CA", "ORD-5520-TX", np.nan])

extracted_df = codes.str.extract(r"ORD-(?P<Order_ID>\d+)-(?P<State>[A-Z]{2})")
display(extracted_df)

,Order_ID,State
0,9821,NY
1,1044,CA
2,5520,TX
3,NaN,NaN


### Q3: Advanced Interview Challenge: High-Risk Restaurant Keywords
**Challenge**: Across all food inspections marked as **'Risk 1 (High)'**, extract the culinary keyword (e.g. `PIZZA`, `GRILL`, `CAFE`, `RESTAURANT`, `TACOS`, `BAKERY`, `LOUNGE`) and find the top 3 most common high-risk establishment types!

In [14]:
# Extract culinary keyword from restaurant Name and find top high-risk categories
pattern = r"\b(PIZZA|CAFE|RESTAURANT|GRILL|TACOS?|BAKERY|LOUNGE)\b"

high_risk_keywords = (
    inspections[inspections["Risk_Category"] == "High"]
    .assign(keyword=lambda df: df["Name"].str.upper().str.extract(pattern, expand=False))
    .dropna(subset=["keyword"])
    .groupby("keyword")["Risk_Category"]
    .count()
    .sort_values(ascending=False)
    .head(3)
)

print("Top 3 High-Risk Restaurant Types:")
display(high_risk_keywords)

Top 3 High-Risk Restaurant Types:


keyword
RESTAURANT    8588
CAFE          3936
GRILL         3839
Name: Risk_Category, dtype: int64